# MIDDLEWARE

middleware provides a way to more tightly control what happens inside the agent.Middleware is useful for the following:

1. tracking agent behavior with logging,analytics,and debugging.
2. transforming prompts ,tool selection,and output formating.
3. adding retries,fallbacks,and early termination logic.
4. applying rate limits,guardrails,and Pll detection

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
model = ChatGroq(model="qwen/qwen3-32b")

print(os.getenv("GROQ_API_KEY"))  # optional check

### Summarization Middleware
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

1. Long-running conversations that exceed context windows.
2. Multi-turn dialogues with extensive history.
3. Applications where preserving full conversation context matters.

In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

### agent with message based summarization

agent= create_agent(
    model="groq:llama-3.1-8b-instant",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)
agent

In [ ]:
## Thread id to run agent
config={"configurable":{"thread_id":"test1"}}

In [ ]:
##  Alternative test data
questions=[
      "what is 2+2?",
      "what is 21*5?",
      "what is 100/4?",
      "what is 34-9?",
      "what is 8*8?",
      "what is 4*34?",
]
for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]},config)
    print(f"Messages:{response}")
    print(f"Messages:{len(response['messages'])}")

## Token size

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long responses."""
    
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night
    2. City Inn - 4 star, $180/night
    3. Budget Stay - 3 star, $75/night
    """


# Create agent WITHOUT middleware
agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "test1"}}


# Approximate token counter
def count_tokens(messages):
    total_char = sum(len(str(m.content)) for m in messages)
    return total_char // 4


# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:

    response = agent.invoke(
        {
            "messages": [
                HumanMessage(content=f"Find hotels in {city}")
            ]
        },
        config=config
    )

    tokens = count_tokens(response["messages"])

    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")

    print(response["messages"])

    print("-" * 50)

### Human-in-the-loop
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

1. High-stakes operations requiring human approval (e.g. database writes, financial transactions).
2. Compliance workflows where human oversight is mandatory.
3. Long-running conversations where human feedback guides the agent.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


# Tool to read email
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    
    return f"Email content for ID: {email_id}"


# Tool to send email
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    
    return f"Email sent to {recipient} with subject '{subject}'"


# Create agent
agent = create_agent(
    model="groq:llama-3.1-8b-instant",

    tools=[
        read_email_tool,
        send_email_tool
    ],

    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": True
            }
        )
    ],

    checkpointer=InMemorySaver()
)
agent